# Lesson04. 机器狗派对！编排你的专属舞步

**教学主题：** 用代码创造复杂的连续动作。

**核心目标：** 掌握 `for` 循环与函数（`function`）的使用。

**课程安排：**

- **前30分钟（新工具）：** 将一套"拜年动作"（连续点头10次+招手）打包成一个叫做 `new_year_greeting()` 的指令。

- **后90分钟（创意编舞）：**
  - **设计舞步：** 学习控制身体姿态的指令（摇摆、扭动），设计2-3个基本舞步。
  - **串联与重复：** 舞步用函数打包，再用循环重复播放，创作一段30秒的机器狗舞蹈。
  - 播放音乐，全场的Go2一起开派对！

## 4.1 导入依赖并初始化客户端

In [1]:
import time  # 时间模块，用于控制延时
import sys   # 系统模块

# 导入宇树SDK通信和运动控制模块
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化通信通道并创建运动控制客户端
ChannelFactoryInitialize(0, "ens37")
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()

## 4.2 使用for循环编排动作队列

`for` 循环可以让一段代码重复执行多次，非常适合让机器狗重复做某个动作。

In [3]:
# 连续旋转：循环4次，每次以1.0 rad/s的角速度旋转，持续1秒
for i in range(4):
    sport_client.Move(0, 0, 1.0)  # 原地逆时针旋转
    time.sleep(1)  # 等待1秒再发送下一条指令

In [4]:
# 连续执行动作：先做2次Scrape（刨地），再做2次Content（撒娇）
for i in range(2):
    sport_client.Scrape()   # 刨地动作
    # sport_client.Stretch()  # 伸展动作（已注释，可取消注释尝试）
    # time.sleep(4)

for i in range(2):
    sport_client.Content()  # 撒娇/开心动作

In [ ]:
# 使用if-else条件判断交替执行不同动作
# i % 2 == 0 表示偶数次执行Scrape，奇数次执行Content
for i in range(4):
    if i % 2 == 0:
        print("刨地动作")
        sport_client.Scrape()
    else:
        print("撒娇动作")
        sport_client.Content()

## 4.3 多动作串联

将多个动作按顺序串联起来，封装成一个函数，实现一套完整的表演流程。

In [ ]:
import time
import sys
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化通信通道和运动控制客户端
ChannelFactoryInitialize(0, "ens37")
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()


def run_robot_actions():
    """按顺序执行一套完整的机器人动作表演"""
    # ---- 第一部分：坐下与起立 ----
    ret = sport_client.Sit()           # 坐下
    print(f"坐下动作执行结果: {ret}")
    time.sleep(3)                       # 等待动作完成

    ret = sport_client.RiseSit()        # 从坐姿起立
    print(f"起立动作执行结果: {ret}")

    # ---- 第二部分：交替表演刨地和撒娇 ----
    for i in range(4):
        if i % 2 == 0:
            print("刨地动作")
            sport_client.Scrape()
        else:
            print("撒娇动作")
            sport_client.Content()
        time.sleep(2)                   # 每个动作间隔2秒

    # ---- 第三部分：高难度动作展示 ----
    sport_client.StandUp()              # 站立
    print("站立")
    time.sleep(3)

    sport_client.FrontPounce()          # 前扑
    print("前扑")
    time.sleep(3)

    sport_client.FrontFlip()            # 前空翻
    print("前空翻")
    time.sleep(3)

    sport_client.Scrape()               # 刨地
    print("刨地")
    time.sleep(3)

    # ---- 第四部分：舞蹈表演 ----
    sport_client.Dance1()               # 舞蹈动作1
    print("舞蹈1")
    time.sleep(3)

    sport_client.Dance2()               # 舞蹈动作2
    print("舞蹈2")
    time.sleep(3)

    # 表演结束，回到站立状态
    sport_client.StandUp()
    time.sleep(2)


if __name__ == "__main__":
    try:
        run_robot_actions()
        print("所有动作执行完成！")
    except Exception as e:
        print(f"执行过程中出现错误: {e}")
        # 出错时尝试让机器人恢复站立
        sport_client.StandUp()
        time.sleep(2)

## 4.4 随机动作组合

从动作库中随机挑选指定数量的动作执行，每次运行效果都不一样！

In [ ]:
import time
import random  # 随机数模块，用于随机选择动作
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化通信通道和运动控制客户端
ChannelFactoryInitialize(0, "ens37")
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()


def run_random_actions(num_actions):
    """
    从动作库中随机选择指定数量的不重复动作并依次执行。
    
    Args:
        num_actions (int): 要执行的动作数量
    """
    # 定义动作列表：(动作名称, 动作函数, 动作后等待时间)
    actions = [
        ("坐下",   sport_client.Sit, 3),
        ("起立",   sport_client.RiseSit, 0),
        ("刨地",   sport_client.Scrape, 3),
        ("撒娇",   sport_client.Content, 2),
        ("站立",   sport_client.StandUp, 3),
        ("前扑",   sport_client.FrontPounce, 3),
        ("前空翻", sport_client.FrontFlip, 3),
        ("舞蹈1",  sport_client.Dance1, 3),
        ("舞蹈2",  sport_client.Dance2, 3),
    ]
    
    # 参数检查
    if num_actions <= 0:
        raise ValueError("动作数量必须为正整数")
    if num_actions > len(actions):
        raise ValueError(f"动作数量不能超过动作库总数（{len(actions)}个）")
    
    # 使用random.sample随机选择不重复的动作
    selected_actions = random.sample(actions, num_actions)
    
    # 依次执行选中的动作
    for action_name, action_func, delay in selected_actions:
        try:
            print(f"执行动作: {action_name}")
            ret = action_func()
            print(f"  返回值: {ret}")
            if delay > 0:
                time.sleep(delay)
        except Exception as e:
            print(f"  执行动作 {action_name} 时出错: {e}")
            time.sleep(1)
    
    print(f"\n{num_actions}个随机动作全部执行完成！")


if __name__ == "__main__":
    # 设置要随机执行的动作数量（可自行修改）
    CUSTOM_NUM_ACTIONS = 5
    
    try:
        run_random_actions(CUSTOM_NUM_ACTIONS)
    except ValueError as ve:
        print(f"参数错误: {ve}")
    except Exception as e:
        print(f"程序执行出错: {e}")
        # 出错时尝试恢复站立
        try:
            sport_client.StandUp()
            time.sleep(3)
        except:
            pass


## 4.5 难度递进式的舞蹈编排

通过嵌套循环和函数参数，创建难度逐步提升的舞蹈序列。


In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化通信通道和运动控制客户端
ChannelFactoryInitialize(0, "ens37")
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()


# ========== 4.5.1 简单舞步：基础抖动 ==========

def basic_shake(repeat=1):
    """
    基础舞步：简单抖动。
    适合初学者，只用两个动作的重复。
    
    Args:
        repeat (int): 抖动重复次数
    """
    print("【基础舞步：抖动】")
    for i in range(repeat):
        sport_client.Scrape()  # 刨地
        time.sleep(1)
        sport_client.Content()  # 撒娇
        time.sleep(1)
    print("✓ 基础舞步完成\n")


# ========== 4.5.2 中级舞步：摇摆组合 ==========

def intermediate_groove(repeat=1):
    """
    中级舞步：坐立交替+撒娇摇摆。
    难度递进，融合更多动作类型。
    
    Args:
        repeat (int): 舞步重复次数
    """
    print("【中级舞步：摇摆组合】")
    for cycle in range(repeat):
        # 坐立交替（2次）
        sport_client.Sit()
        time.sleep(1)
        sport_client.RiseSit()
        time.sleep(1)
        
        # 摇摆舞蹈（刨地+撒娇+刨地）
        sport_client.Scrape()
        time.sleep(1)
        sport_client.Content()
        time.sleep(1)
        sport_client.Scrape()
        time.sleep(1)
        
        print(f"  ✓ 第{cycle+1}轮舞步完成")
    print("✓ 中级舞步完成\n")


# ========== 4.5.3 高级舞步：跳跃+翻滚 ==========

def advanced_acrobatics():
    """
    高级舞步：前扑+前空翻组合。
    难度最高，展示机器狗的极限能力。
    """
    print("【高级舞步：跳跃+翻滚】")
    
    # 准备动作：站立
    sport_client.StandUp()
    print("  → 站立准备")
    time.sleep(2)
    
    # 前扑
    sport_client.FrontPounce()
    print("  → 前扑！")
    time.sleep(3)
    
    # 站立恢复
    sport_client.StandUp()
    time.sleep(2)
    
    # 前空翻
    sport_client.FrontFlip()
    print("  → 前空翻！")
    time.sleep(3)
    
    # 最后摆动一次
    sport_client.Scrape()
    print("  → 最后一次刨地")
    time.sleep(1)
    
    # 回到站立
    sport_client.StandUp()
    print("✓ 高级舞步完成\n")


# ========== 4.5.4 难度递进式演示 ==========

def progressive_dance_demo():
    """
    演示难度递进：从简单到复杂的舞蹈序列。
    展示代码复用、参数化和嵌套循环的强大威力。
    """
    print("\n" + "="*60)
    print("【难度递进式舞蹈演示】")
    print("="*60 + "\n")
    
    # 第一阶段：基础（重复3次）
    print("【第一阶段：基础难度】")
    basic_shake(repeat=2)
    time.sleep(2)
    
    # 第二阶段：中级（重复2次）
    print("【第二阶段：中级难度】")
    intermediate_groove(repeat=1)
    time.sleep(2)
    
    # 第三阶段：高级（一次性）
    print("【第三阶段：高级难度】")
    advanced_acrobatics()
    time.sleep(2)
    
    print("="*60)
    print("✓ 难度递进式舞蹈演示完成！")
    print("="*60)


# 执行演示
if __name__ == "__main__":
    try:
        progressive_dance_demo()
    except Exception as e:
        print(f"执行出错: {e}")
        sport_client.StandUp()
        time.sleep(2)


## 4.6 创意主题舞蹈演示

根据不同的主题编排不同风格的舞蹈，展现机器狗的多面性。


In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化
ChannelFactoryInitialize(0, "ens37")
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()


# ========== 4.6.1 拜年舞蹈 ==========

def lunar_new_year_dance():
    """
    拜年舞蹈：表达新年祝福和欢乐。
    包含点头、拜年、摇摆等元素。
    """
    print("\n🧧 【拜年舞蹈】🧧\n")
    
    # 准备：站立
    sport_client.StandUp()
    print("✨ 站立 - 恭迎新年")
    time.sleep(2)
    
    # 拜年：3次坐立 = 3个拜年
    for i in range(3):
        sport_client.Sit()
        print(f"🙏 拜年 {i+1}/3...")
        time.sleep(1)
        sport_client.RiseSit()
        time.sleep(1)
    
    # 欢乐摇摆
    print("🎉 欢乐摇摆")
    for i in range(3):
        sport_client.Scrape()
        time.sleep(1)
        sport_client.Content()
        time.sleep(1)
    
    # 结尾：跳跃表达喜悦
    sport_client.StandUp()
    print("🎊 恭喜发财！")
    time.sleep(1)
    
    print("✓ 拜年舞蹈完成\n")


# ========== 4.6.2 摇滚舞蹈 ==========

def rock_dance():
    """
    摇滚舞蹈：激烈、节奏感强。
    充满动感的快速动作组合。
    """
    print("\n🎸 【摇滚舞蹈】🎸\n")
    
    # 摇滚前奏：激烈的坐立交替
    print("⚡ 摇滚前奏")
    for i in range(5):
        sport_client.Sit()
        time.sleep(0.5)
        sport_client.RiseSit()
        time.sleep(0.5)
    
    time.sleep(1)
    
    # 摇滚舞动：快速的刨地+撒娇
    print("🔥 摇滚主段")
    for i in range(4):
        sport_client.Scrape()
        time.sleep(0.7)
        sport_client.Content()
        time.sleep(0.7)
    
    time.sleep(1)
    
    # 摇滚高潮：高难度动作
    print("🚀 摇滚高潮")
    sport_client.StandUp()
    time.sleep(1)
    sport_client.FrontPounce()
    print("  💥 前扑！")
    time.sleep(2)
    
    sport_client.StandUp()
    time.sleep(1)
    
    print("✓ 摇滚舞蹈完成\n")


# ========== 4.6.3 芭蕾舞蹈 ==========

def ballet_dance():
    """
    芭蕾舞蹈：优雅、缓慢、连贯。
    强调身体的流畅性和美感。
    """
    print("\n🩰 【芭蕾舞蹈】🩰\n")
    
    # 开场：站立准备
    sport_client.StandUp()
    print("✨ 优雅站立")
    time.sleep(2)
    
    # 第一部分：缓慢的坐立 (优雅蹲下)
    print("🎼 第一乐章：缓慢蹲下")
    for i in range(2):
        sport_client.Sit()
        time.sleep(2)  # 缓慢进行
        sport_client.RiseSit()
        time.sleep(2)
    
    time.sleep(1)
    
    # 第二部分：柔和的摇摆
    print("🎼 第二乐章：柔和摇摆")
    for i in range(3):
        sport_client.Scrape()
        time.sleep(1.5)
        sport_client.Content()
        time.sleep(1.5)
    
    # 结尾：优雅收尾
    print("🎭 华丽谢幕")
    sport_client.StandUp()
    time.sleep(2)
    
    print("✓ 芭蕾舞蹈完成\n")


# ========== 4.6.4 街舞 ==========

def street_dance():
    """
    街舞：融合拍子感、创意动作的现代舞蹈。
    展现机器狗的潮流风范。
    """
    print("\n👟 【街舞】👟\n")
    
    print("🎤 街舞 DROP！")
    time.sleep(1)
    
    # 开局：坐立 = 底层动作
    sport_client.Sit()
    print("↓ 蹲下")
    time.sleep(1)
    
    # 爆发：站立 + 快速动作
    sport_client.RiseSit()
    print("↑ 弹起")
    time.sleep(0.5)
    
    # 高速节奏
    print("🎵 高速节奏")
    for i in range(3):
        sport_client.Scrape()
        time.sleep(0.6)
        sport_client.Content()
        time.sleep(0.6)
    
    time.sleep(1)
    
    # 街舞特技：跳跃
    print("✨ 街舞特技")
    sport_client.StandUp()
    time.sleep(1)
    sport_client.FrontFlip()
    print("  🌟 翻滚！")
    time.sleep(2)
    
    sport_client.StandUp()
    print("🎤 街舞完成！")
    time.sleep(1)
    
    print("✓ 街舞结束\n")


# ========== 4.6.5 创意舞蹈汇演 ==========

def creative_dance_showcase():
    """
    创意舞蹈汇演：展示多种舞蹈风格。
    """
    print("\n" + "="*60)
    print("【创意舞蹈主题汇演】")
    print("="*60)
    
    # 表演1：拜年舞
    lunar_new_year_dance()
    time.sleep(3)
    
    # 表演2：摇滚舞
    rock_dance()
    time.sleep(3)
    
    # 表演3：芭蕾舞
    ballet_dance()
    time.sleep(3)
    
    # 表演4：街舞
    street_dance()
    
    print("="*60)
    print("✓ 创意舞蹈汇演圆满结束！")
    print("="*60)


# 执行汇演
if __name__ == "__main__":
    try:
        creative_dance_showcase()
    except Exception as e:
        print(f"执行出错: {e}")
        sport_client.StandUp()
        time.sleep(2)

In [ ]:

## 4.7 舞蹈组合与节奏控制

通过精确控制时间间隔，实现不同节奏的舞蹈效果。


In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化
ChannelFactoryInitialize(0, "ens37")
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()


# ========== 4.7.1 BPM节奏控制 ==========

class BPMDanceController:
    """
    基于BPM（每分钟节拍数）的舞蹈节奏控制器。
    可以精确控制舞蹈的节奏速度。
    """
    
    def __init__(self, bpm):
        """
        初始化BPM控制器。
        
        Args:
            bpm (int): 每分钟节拍数 (Beats Per Minute)
        """
        self.bpm = bpm
        # 计算每个四分音符的时长（秒）
        self.beat_duration = 60.0 / bpm
    
    def wait_beat(self, beats=1):
        """等待指定数量的节拍"""
        time.sleep(self.beat_duration * beats)
    
    def get_beat_duration(self):
        """获取每个节拍的时长"""
        return self.beat_duration


# ========== 4.7.2 缓慢舞蹈（BPM = 60） ==========

def slow_dance():
    """
    缓慢舞蹈：BPM=60，适合冥想和优雅表演。
    每个动作有充足的反应时间。
    """
    print("\n🌙 【缓慢舞蹈】(BPM=60) 🌙\n")
    
    controller = BPMDanceController(bpm=60)
    
    # 准备
    sport_client.StandUp()
    print("站立准备...")
    controller.wait_beat(2)  # 等待2个节拍
    
    # 缓慢坐立
    print("缓慢蹲下...")
    for i in range(3):
        sport_client.Sit()
        controller.wait_beat(2)
        sport_client.RiseSit()
        controller.wait_beat(2)
    
    # 缓慢摇摆
    print("柔和摇摆...")
    for i in range(4):
        sport_client.Scrape()
        controller.wait_beat(1.5)
        sport_client.Content()
        controller.wait_beat(1.5)
    
    print("✓ 缓慢舞蹈完成\n")


# ========== 4.7.3 正常舞蹈（BPM = 120） ==========

def normal_dance():
    """
    正常舞蹈：BPM=120，标准节奏，适合大众舞蹈。
    """
    print("\n🎵 【正常舞蹈】(BPM=120) 🎵\n")
    
    controller = BPMDanceController(bpm=120)
    
    # 准备
    sport_client.StandUp()
    print("准备就绪...")
    controller.wait_beat(2)
    
    # 正常坐立
    print("坐立交替...")
    for i in range(4):
        sport_client.Sit()
        controller.wait_beat(1)
        sport_client.RiseSit()
        controller.wait_beat(1)
    
    # 正常摇摆
    print("标准摇摆...")
    for i in range(4):
        sport_client.Scrape()
        controller.wait_beat(0.75)
        sport_client.Content()
        controller.wait_beat(0.75)
    
    print("✓ 正常舞蹈完成\n")


# ========== 4.7.4 快速舞蹈（BPM = 180） ==========

def fast_dance():
    """
    快速舞蹈：BPM=180，高能节奏，展现速度和激情。
    """
    print("\n⚡ 【快速舞蹈】(BPM=180) ⚡\n")
    
    controller = BPMDanceController(bpm=180)
    
    # 准备
    sport_client.StandUp()
    print("准备冲刺！")
    controller.wait_beat(1)
    
    # 快速坐立
    print("闪电坐立...")
    for i in range(6):
        sport_client.Sit()
        controller.wait_beat(0.5)
        sport_client.RiseSit()
        controller.wait_beat(0.5)
    
    # 快速摇摆
    print("飞快摇摆...")
    for i in range(6):
        sport_client.Scrape()
        controller.wait_beat(0.5)
        sport_client.Content()
        controller.wait_beat(0.5)
    
    # 高潮：跳跃
    print("高潮时刻！")
    sport_client.StandUp()
    controller.wait_beat(0.5)
    sport_client.FrontPounce()
    print("  💥 前扑！")
    controller.wait_beat(2)
    
    print("✓ 快速舞蹈完成\n")


# ========== 4.7.5 节奏对比演示 ==========

def rhythm_comparison_demo():
    """
    展示不同BPM下的舞蹈差异，帮助理解节奏的重要性。
    """
    print("\n" + "="*60)
    print("【不同节奏的舞蹈对比演示】")
    print("="*60)
    print("通过改变BPM（每分钟节拍数），体验舞蹈节奏的变化\n")
    
    # 演示1：缓慢
    slow_dance()
    time.sleep(2)
    
    # 演示2：正常
    normal_dance()
    time.sleep(2)
    
    # 演示3：快速
    fast_dance()
    
    print("="*60)
    print("✓ 节奏对比演示完成！")
    print("="*60)


# ========== 4.7.6 变速舞蹈 ==========

def accelerating_dance():
    """
    加速舞蹈：从慢到快的渐进式加速。
    展现舞蹈节奏的动态变化。
    """
    print("\n📈 【加速舞蹈】📈\n")
    
    # 起始速度：缓慢（BPM=80）
    bpm_values = [80, 100, 120, 140, 160, 180]
    
    for bpm in bpm_values:
        controller = BPMDanceController(bpm=bpm)
        print(f"BPM = {bpm}...")
        
        # 每个速度做2个动作组合
        for j in range(2):
            sport_client.Scrape()
            controller.wait_beat(0.5)
            sport_client.Content()
            controller.wait_beat(0.5)
    
    print("\n⚡ 达到最高速！")
    time.sleep(1)
    
    # 结尾：跳跃
    sport_client.StandUp()
    time.sleep(0.5)
    sport_client.FrontFlip()
    print("🌟 前空翻完成！")
    time.sleep(2)
    
    sport_client.StandUp()
    print("✓ 加速舞蹈完成\n")


# 执行演示
if __name__ == "__main__":
    try:
        # 选择演示内容（可注释切换）
        print("选择演示模式：")
        print("1. 节奏对比演示（推荐）")
        print("2. 加速舞蹈")
        
        # 默认执行节奏对比演示
        rhythm_comparison_demo()
        
        # 可选：再执行加速舞蹈
        # time.sleep(3)
        # accelerating_dance()
        
    except Exception as e:
        print(f"执行出错: {e}")
        sport_client.StandUp()
        time.sleep(2)


## 4.8 综合派对表演

集合所有学到的知识，创作一场完整的机器狗派对表演。


In [ ]:
import time
import random
from unitree_sdk2py.core.channel import ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.go2.sport.sport_client import SportClient

# 初始化
ChannelFactoryInitialize(0, "ens37")
sport_client = SportClient()
sport_client.SetTimeout(10.0)
sport_client.Init()


# ========== 4.8.1 派对开场 ==========

def party_opening():
    """派对开场：打招呼和热身"""
    print("\n" + "🎉"*30)
    print("【机器狗派对正式开始！】")
    print("🎉"*30)
    
    print("\n👋 欢迎各位嘉宾！")
    
    # 站立准备
    sport_client.StandUp()
    print("🐕 机器狗准备登场...")
    time.sleep(2)
    
    # 热身：快速坐立
    print("\n🔥 热身动作")
    for i in range(3):
        sport_client.Sit()
        time.sleep(0.5)
        sport_client.RiseSit()
        time.sleep(0.5)
    
    print("✓ 热身完成，派对火热进行中！\n")
    time.sleep(1)


# ========== 4.8.2 派对主舞台 ==========

def party_main_show():
    """派对主舞台：展现各种舞蹈风格"""
    print("【主舞台表演】\n")
    
    # 第一幕：摇摆舞
    print("第一幕：疯狂摇摆！")
    for i in range(5):
        sport_client.Scrape()
        time.sleep(0.7)
        sport_client.Content()
        time.sleep(0.7)
    print("✓ 摇摆舞完成！\n")
    time.sleep(1)
    
    # 第二幕：坐立律动
    print("第二幕：坐立律动！")
    for i in range(4):
        sport_client.Sit()
        time.sleep(0.8)
        sport_client.RiseSit()
        time.sleep(0.8)
    print("✓ 坐立律动完成！\n")
    time.sleep(1)
    
    # 第三幕：高难度动作
    print("第三幕：高难度特技！")
    sport_client.StandUp()
    time.sleep(1)
    
    print("  → 前扑！")
    sport_client.FrontPounce()
    time.sleep(2)
    
    sport_client.StandUp()
    time.sleep(1)
    
    print("  → 前空翻！")
    sport_client.FrontFlip()
    time.sleep(2)
    
    print("✓ 高难度特技完成！\n")
    time.sleep(1)


# ========== 4.8.3 互动环节 ==========

def interactive_segment():
    """与观众的互动：随机表演"""
    print("【互动环节：观众选择舞蹈】\n")
    
    dance_moves = [
        ("舞蹈1", sport_client.Dance1),
        ("舞蹈2", sport_client.Dance2),
        ("前扑", sport_client.FrontPounce),
    ]
    
    # 随机选择3个舞蹈表演
    selected = random.sample(dance_moves, min(3, len(dance_moves)))
    
    for i, (name, move) in enumerate(selected, 1):
        print(f"表演 {i}: {name}")
        sport_client.StandUp()
        time.sleep(0.5)
        move()
        time.sleep(2)
    
    print("✓ 互动环节完成！\n")
    time.sleep(1)


# ========== 4.8.4 派对高潮 ==========

def party_climax():
    """派对高潮：最激烈的表演"""
    print("【派对高潮：终极表演】\n")
    print("⚡ 准备迎接高潮...⚡\n")
    
    # 快速节奏舞蹈
    print("快速摇摆...")
    for i in range(8):
        sport_client.Scrape()
        time.sleep(0.5)
        sport_client.Content()
        time.sleep(0.5)
    
    time.sleep(1)
    
    # 快速坐立
    print("疯狂跳跃...")
    for i in range(6):
        sport_client.Sit()
        time.sleep(0.4)
        sport_client.RiseSit()
        time.sleep(0.4)
    
    time.sleep(1)
    
    # 连续高难度动作
    print("连续特技表演！")
    sport_client.StandUp()
    time.sleep(0.5)
    
    # 前扑
    sport_client.FrontPounce()
    print("  💥 前扑！")
    time.sleep(1)
    
    sport_client.StandUp()
    time.sleep(0.5)
    
    # 前空翻
    sport_client.FrontFlip()
    print("  🌟 前空翻！")
    time.sleep(2)
    
    # 最后的疯狂摇摆
    print("最后的疯狂...")
    for i in range(5):
        sport_client.Scrape()
        time.sleep(0.3)
        sport_client.Content()
        time.sleep(0.3)
    
    print("✓ 高潮完成！全场沸腾！\n")


# ========== 4.8.5 派对谢幕 ==========

def party_finale():
    """派对谢幕：感谢观众，回到站立"""
    print("【派对谢幕】\n")
    
    # 几次鞠躬/点头表示感谢
    print("感谢各位观众的热烈掌声！")
    
    sport_client.StandUp()
    time.sleep(1)
    
    # 3次坐立表示鞠躬
    for i in range(3):
        sport_client.Sit()
        print("  🙏 谢谢你们！")
        time.sleep(1)
        sport_client.RiseSit()
        time.sleep(0.5)
    
    # 最后一次优雅的站立
    sport_client.StandUp()
    print("\n✨ 派对完美结束！✨")
    print("我们下次派对再见！\n")


# ========== 4.8.6 完整派对表演 ==========

def complete_party_show():
    """
    完整的派对表演流程。
    包含开场、主舞台、互动、高潮、谢幕等环节。
    """
    try:
        # 开场
        party_opening()
        time.sleep(2)
        
        # 主舞台
        party_main_show()
        time.sleep(2)
        
        # 互动环节
        interactive_segment()
        time.sleep(2)
        
        # 派对高潮
        party_climax()
        time.sleep(2)
        
        # 谢幕
        party_finale()
        
        print("="*60)
        print("🎊 机器狗派对圆满成功！🎊")
        print("="*60)
        
    except Exception as e:
        print(f"\n❌ 派对中发生错误: {e}")
        print("尝试恢复...")
        sport_client.StandUp()
        time.sleep(2)


# 执行派对
if __name__ == "__main__":
    complete_party_show()